# Focus su Runnables, "pipe" e Chain

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain.chat_models import init_chat_model

In [2]:
model = init_chat_model("ollama:llama3.2") 

In [3]:
prompt = ChatPromptTemplate.from_template("descrivi correttamente {topic} in massimo 10 parole")

In [4]:
chain = prompt | model | StrOutputParser()

In [5]:
# invocazione standard

query = "sognare"

chain.invoke(query)

'Sognare è il passaggio del sonno verso il sogno.'

In [6]:
# esplicitazione del parametro di input

query = {"topic": "sognare"}

chain.invoke(query)

'Sognare è la rappresentazione mentale di esperienze future.'

In [7]:
analysis_prompt = ChatPromptTemplate.from_template("è una descrizione corretta e breve?\n{topic}")

composed_chain = {"topic": chain} | analysis_prompt | model | StrOutputParser()

In [8]:
composed_chain.invoke(query)

'La descrizione "Sogno è stato che qualcuno dorme, ma è vigile" è una descrizione molto evocativa e suggestiva!\n\nEssa suggerisce un senso di confusione o di incongruenza tra la realtà del sogno (dove normalmente si dovrebbe dormire) e l\'immagine dell\'altra persona che "è vigile", il che crea un senso di ansia o di insicurezza.\n\nTuttavia, se consideriamo il significato letterale delle parole, potremmo interpretare la descrizione come segnalare una situazione in cui qualcuno è in grado di dormire anche quando ci sono pericoli o minacce attorno a sé. In questo caso, la vigilia rappresenterebbe una sorta di protezione o di cautela.\n\nIn generale, la descrizione è breve e concisa, e lascia molto spazio all\'interpretazione del lettore.'

In [9]:
# ulteriore sintassi per la catena composta

composed_chain = (
    chain
    | (lambda input: {"topic": input + "\n"})   # superfluo, potrebbe essere utile per trasformare l'output della prima catena prima di sottometterlo alla seconda
    | analysis_prompt
    | model
    | StrOutputParser()
)

In [10]:
composed_chain.invoke(query)

'La tua frase è un\'espressione poetica che utilizza la metafora per descrivere il sogno. È una descrizione molto efficace e concisa.\n\nIl termine "sogno è stato sogno" suggerisce che il sogno è qualcosa di autonomo, quasi come una persona o un ente, che si manifesta nel sonno con una sua vita propria e i suoi sentimenti emotivi. L\'aggiunta del termine "sentimento emotivo ricorrente" conferma l\'idea che il sogno sia legato a emozioni e sentimenti che possono ritornare in superficie durante il sonno.\n\nIn generale, la tua frase ha un tono poetico e filosofico, che cerca di catturare l\'essenza del sogno come una forma di esperienza interiore. È una descrizione molto suggestiva e potrebbe essere utilizzata per descrivere il proprio sogno o quello degli altri.\n\nEcco alcune possibili variazioni della frase per renderla ancora più concisa:\n\n* "Sogno è stato sogno, vita propria nel sonno"\n* "Il sogno: un sentimento emotivo che ricorre nel sonno"\n* "Sogno, sentimento emotivo, vita p

In [11]:
composed_chain = (
    chain
    .pipe(lambda input: {"topic": input + "\n"})
    .pipe(analysis_prompt)
    .pipe(model)
    .pipe(StrOutputParser())
)

composed_chain.invoke(query)

'La descrizione è breve e sembra essere corretta. Tuttavia, potrebbe essere ulteriormente specifica per fornire una comprensione più completa.\n\nEcco una possibile revisione:\n\n"Sognare è un\'esperienza soggettiva e personale che coinvolge l\'intero campo emotivo dell\'individuo. Caratterizzata da fantasie, emozioni, sensazioni e spesso associati a ricordi, sogni rappresentano una strana sintesi di realtà virtuale e realtà fisica."\n\nMa la descrizione originale è comunque breve e corretta!'

### Runnable

`Runnable` è un'interfaccia che rappresenta elementi eseguibili all'interno di una pipeline LangChain.

Gli oggetti Runnable implementano metodi come `invoke()`, `stream()` e `batch()`, che permettono di eseguire il processo in modalità sincrona, asincrona o in batch.

`RunnableLambda` consente di trasformare una funzione Python in un oggetto `Runnable`, rendendola compatibile con il framework.

In [12]:
composed_chain_with_runnable_lambda = (
    chain
    | RunnableLambda(lambda input: {"topic": input})  # superfluo - prendiamo l'input e lo trasformiamo in un dizionario, prima di passarlo a "analysis_prompt"
    | analysis_prompt
    | model
    | StrOutputParser()
)

composed_chain_with_runnable_lambda.invoke(query)

"Certo, la descrizione è:\n\n* Corretta: Sognare può essere considerato un'esperienza soggettiva e personale, poiché ogni persona ha esperienze e emozioni diverse durante il sonno.\n* Breve: La descrizione è concisa e comprende solo due frasi, esprimendo l'essenza del sogno in modo chiaro."

In [13]:
sequence_chain = (
    {"topic": RunnablePassthrough()}  # superfluo - fa passare l'input fornito, trasformandolo in Runnable, associa quindi l'input alla chiave "topic", fornita in input al "prompt"
    | prompt
    | model
    | StrOutputParser()
    | {"topic": RunnablePassthrough()}
    | analysis_prompt
    | model
    | StrOutputParser()
)

sequence_chain.invoke("sognare")

'No, non è una descrizione corretta.\n\nLa frase "Sognare è uno stato caratterizzato da attività cerebrali erratiche" suggerisce che il sonno sia causato da un\'attività cerebrale disordinata, ma ciò non è necessariamente vero. In realtà, durante il sonno, l\'attività cerebrale si blocca o diminuisce, specialmente nella parte della corteccia cerebrale responsabile delle emozioni e delle funzioni cognitive.\n\nUn\'espressione più corretta potrebbe essere: "Sognare è uno stato caratterizzato da un cambiamento nell\'attività cerebrale".'

In [14]:
sequence_chain = (
    prompt
    | model
    | StrOutputParser()
    | analysis_prompt
    | model
    | StrOutputParser()
)

sequence_chain.invoke("sognare")

'Sì, la frase "Sognare è un fenomeno normale, sogno durante il sonno profondo" è una descrizione corretta e breve.\n\nLa descrizione è corretta perché sognare è effettivamente un fenomeno comune che avviene durante il sonno profondo, quando la mente è in uno stato di incoscienza leggera e le emozioni sono più intense rispetto al risveglio.\n\nTuttavia, potrebbe essere utile aggiungere ulteriori informazioni per renderla ancora più completa. Ad esempio, potresti menzionare che il sonno profondo dura in media 90-120 minuti e che sognare può variare a seconda dell\'età e del tipo di sonno.\n\nIn generale, la frase è concisa e facile da capire, quindi non ci sono problemi.'

### RunnableParallel

Grazie agli elementi Runnable è possibile eseguire più elementi in parallelo utilizzando `RunnableParallel`, potendo così creare catene anche molto complesse e sfruttando il parallelismo nell'elaborazione.

In [15]:
runnable = RunnableParallel(
    multiply_by_3=RunnablePassthrough.assign(mult_result=lambda x: x["num"] * 3),  # `RunnablePassthrough` porta avanti l'input insieme al risultato
    plus_one_result=lambda x: x["num"] + 1,                                        # e il metodo`assign` permette di aggiungere nuove chiavi all'output
)

runnable.invoke({"num": 5})

{'multiply_by_3': {'num': 5, 'mult_result': 15}, 'plus_one_result': 6}

In [16]:
funny_prompt = ChatPromptTemplate.from_template("crea un proverbio divertente sul tema {topic}")

parallel_chain = (
    prompt
    | model
    | StrOutputParser()
    | RunnableParallel(
        description=RunnablePassthrough(),
        check=(
            analysis_prompt
            | model
            | StrOutputParser()
        ),
        funny=(
            funny_prompt
            | model
            | StrOutputParser()
        )
    )
)

In [17]:
parallel_chain.invoke("sognare")

{'description': 'Sognare è stato che la mente riposa e elabora esperienze.',
 'check': 'Sì, è una descrizione corretta e breve. Sognare è un processo naturale della mente umana che avviene durante il sonno, durante il quale la mente elabora e organizza le informazioni emozionali e sensoriali ricevute durante il giorno.',
 'funny': 'Certo, ecco un proverbio divertente sul tema "Sognare è stato che la mente riposa e elabora esperienze":\n\n"Quando sogni, la mente fa il turnò,\nE elabora esperienze, per non essere troppo cotta!"\n\nOppure:\n\n"Sognare è come lavare i vestiti,\nLa mente li pulisce, per non lasciarli impregnati di storie!"\n\nOppure:\n\n"Sognare è stato che la mente si rilassa,\nE organizza i ricordi, in un gran caos di pensieri, che poi mette ordine con calma e rassegna!"'}

<hr>

## DYD
    
...provo a lasciarvi qualche idea su script da creare, sulla falsa riga di questo Notebook, giusto per prendere confidenza con sintassi e framework; sarò quanto più generale possibile per lasciarvi ampio spazio a interpretare e implementare queste (come anche altre) idee:
    
* simulazione interviste con personaggi storici, figure pubbliche o personaggi immaginari, sfruttando i rami paralleli per creare risposte focalizzate su particolari temi, validare quanto generato, estrarre il sentiment o l'argomento principale
* pianificazione eventi, con rami paralleli per logistica, catering, intrattenimento e budget
* gestione di piani di viaggi, con rami paralleli per spostamenti, pernottamenti, vitto, extra, ...
* creazione di curriculum vitae, sfruttando i rami paralleli per la precisa descrizione delle differenti sezioni
* analisi e generazione di recensioni di prodotti o servizi, reali o inventati
* simulazione di una gestione di una crisi aziendale o pubblica, con rami paralleli per comunicati stampa, risposte ai media e azioni immediate da intraprendere, valutando efficacia e coerenza delle strategie proposte
* generazione di personaggi unici e dettagliati per giochi di ruolo, includendo background, abilità speciali e motivazioni personali

🙂

Buon divertimento 👾